# Chapter 11: Pandas 

We saw in Chapter 9 how to import data from comma separated files (.csv) or other text files into numpy ndarrays with 1 or 2 axes, and how we can manipulate this data, add rows, columns etc.
One of the flaws of this approach is that it is really easy to make mistakes. While the data we read is structured (each row consisted of last name, first name, student id and several grades), we had to remember that midterm1 was a certain column, midterm2 a different one etc. These column numbers had no real meaning, and the order in which they were labelled was irrelevant. Basically, we have imported the *data* but lost the *meta-data*.

Maybe it would have been better to create a dictionary whose keys are the grade items and values list of grades:

In [1]:
import numpy as np
grades  = np.loadtxt('grades.csv',delimiter = ',', skiprows= 1, usecols=(3, 4, 5, 6, 7, 8, 9), max_rows=10)
# Create a dictionary of lists from this data:
keys = ("Quizzes average", "HW1 mark", "HW2 mark", "HW3 mark", "Exam1 mark", "Exam2 mark", "Final exam mark")
gradesD = {}
for col, k in enumerate(keys):
    print(col, k)
    gradesD[k] = grades[:,col]


0 Quizzes average
1 HW1 mark
2 HW2 mark
3 HW3 mark
4 Exam1 mark
5 Exam2 mark
6 Final exam mark


This makes operations on columns really easy:

In [2]:
gradesD['HW average'] = (gradesD['HW1 mark'] + gradesD['HW3 mark'] + gradesD['HW3 mark']) / 3

Operations on rows are a bit painful. Say we want all grades of Khan Vitaly with ID 7943944, we need to figure out that they are the 2nd students in the initial file

In [3]:
row = 2
record = [gradesD[k][row] for k in gradesD.keys()]
print(record)

[np.float64(92.0), np.float64(79.0), np.float64(85.0), np.float64(92.0), np.float64(76.0), np.float64(60.0), np.float64(87.0), np.float64(87.66666666666667)]


In retrospect, maybe we should have created a list of dictionaries:


In [4]:
gradesL = []
for row in grades:
    record = {}
    for col, k in enumerate(keys):
        record[k] = row[col]
    gradesL.append(record)
print(gradesL)

[{'Quizzes average': np.float64(96.0), 'HW1 mark': np.float64(77.0), 'HW2 mark': np.float64(82.0), 'HW3 mark': np.float64(91.0), 'Exam1 mark': np.float64(88.0), 'Exam2 mark': np.float64(78.0), 'Final exam mark': np.float64(91.0)}, {'Quizzes average': np.float64(94.0), 'HW1 mark': np.float64(76.0), 'HW2 mark': np.float64(87.0), 'HW3 mark': np.float64(90.0), 'Exam1 mark': np.float64(90.0), 'Exam2 mark': np.float64(67.0), 'Final exam mark': np.float64(88.0)}, {'Quizzes average': np.float64(92.0), 'HW1 mark': np.float64(79.0), 'HW2 mark': np.float64(85.0), 'HW3 mark': np.float64(92.0), 'Exam1 mark': np.float64(76.0), 'Exam2 mark': np.float64(60.0), 'Final exam mark': np.float64(87.0)}, {'Quizzes average': np.float64(90.0), 'HW1 mark': np.float64(75.0), 'HW2 mark': np.float64(78.0), 'HW3 mark': np.float64(88.0), 'Exam1 mark': np.float64(90.0), 'Exam2 mark': np.float64(67.0), 'Final exam mark': np.float64(84.0)}, {'Quizzes average': np.float64(88.0), 'HW1 mark': np.float64(76.0), 'HW2 mark':

But now, operation on columns are difficult, and even looking for a student is painful...

The problem is that while dictionary are good to represent data with key:value structure, they are not really designed to represent data whose structure is more complex.

This is where Pandas comes into play (note that there are other approaches, including numpy structured arrays).

## 11.1 Pandas `Series`
*Series* are another type of container that can store data of various type. Series can be many things, single data (int, str, ...), lists, nparrays, or dictionaries.


In [5]:
import pandas as pd
rowL0 = pd.Series(grades[0])
rowL1 = pd.Series(grades[1])
print(rowL0)
print(rowL0[1])

0    96.0
1    77.0
2    82.0
3    91.0
4    88.0
5    78.0
6    91.0
dtype: float64
77.0


In [6]:
rowD0 = pd.Series(gradesL[0])
rowD1 = pd.Series(gradesL[1])
print(rowD0)
print(rowD0['HW1 mark'])

Quizzes average    96.0
HW1 mark           77.0
HW2 mark           82.0
HW3 mark           91.0
Exam1 mark         88.0
Exam2 mark         78.0
Final exam mark    91.0
dtype: float64
77.0


Series can be converted to dictionaries, lists, numpy arrays etc. Just like ndarras, operations on Series are vectorized:

In [7]:
print(rowD0.to_list())

[96.0, 77.0, 82.0, 91.0, 88.0, 78.0, 91.0]


In [8]:
print(rowL1.to_dict())

{0: 94.0, 1: 76.0, 2: 87.0, 3: 90.0, 4: 90.0, 5: 67.0, 6: 88.0}


In [9]:
print(rowL0 + rowL1)

0    190.0
1    153.0
2    169.0
3    181.0
4    178.0
5    145.0
6    179.0
dtype: float64


Pandas is actually pretty good at vectorizing operations, even when keys don't match exactly:

In [10]:
rowD1['avg'] = 99
print(rowD0+rowD1)

Exam1 mark         178.0
Exam2 mark         145.0
Final exam mark    179.0
HW1 mark           153.0
HW2 mark           169.0
HW3 mark           181.0
Quizzes average    190.0
avg                  NaN
dtype: float64


Finally, Series can be given a *name*. Here, maybe the student number could have been the name. Or maybe the students names could be the names. We'll see in a bit why. 

In [11]:
print(rowD0.name)
rowD0.name = '620565656'
rowD1.name = '107856531'
rowL0.name = '620565656'
rowL1.name = '107856531'
print(rowD0.name)
print(rowD0)

None
620565656
Quizzes average    96.0
HW1 mark           77.0
HW2 mark           82.0
HW3 mark           91.0
Exam1 mark         88.0
Exam2 mark         78.0
Final exam mark    91.0
Name: 620565656, dtype: float64
